# D4Explainer — Colab Setup
**Runtime:** GPU (T4 or better) — set via *Runtime > Change runtime type*

In [ ]:
# Cell 1: Upload and unzip code
from google.colab import files
import zipfile, os

print('Select D4Explainer.zip when the picker opens...')
uploaded = files.upload()  # select D4Explainer.zip

with zipfile.ZipFile('D4Explainer.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/D4Explainer')
print('CWD:', os.getcwd())

In [ ]:
# Cell 2: Upload and unzip datasets
# Upload one zip per dataset — repeat for each: MUTAG, BA3, BA_shapes, etc.
from google.colab import files
import zipfile, os

os.makedirs('/content/D4Explainer/data', exist_ok=True)

print('Select dataset zip(s) when the picker opens...')
uploaded = files.upload()  # can select multiple files

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/D4Explainer/data/')
        print(f'Extracted {fname}')

print('\ndata/ contents:', os.listdir('/content/D4Explainer/data'))

In [ ]:
# Cell 3: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — check Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3b: Prevent TensorFlow GPU pre-allocation and verify clean GPU state
# TF is installed in Colab alongside PyTorch and grabs all GPU memory by default.
# Must run this before any training — if GPU memory looks wrong, restart the runtime first.
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import torch
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM : {total:.1f} GiB')
print(f'Allocated  : {allocated:.2f} GiB  (should be ~0 on a fresh runtime)')
print(f'Reserved   : {reserved:.2f} GiB')
if allocated > 1.0:
    print('WARNING: >1 GiB already allocated — restart the runtime to clear leftover memory.')

In [ ]:
# Cell 4: Install dependencies
# Note: torch-scatter/sparse/cluster/spline-conv are pre-installed on modern Colab runtimes.
# torch-geometric version pin dropped — 2.4.0 has no wheel for Python 3.12+.
# pandas/networkx version pins dropped — pinned versions predate Python 3.12.
# rdkit-pypi renamed to rdkit.
import torch, subprocess

TORCH_VERSION = torch.__version__.split('+')[0]
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_str = result.stdout.split('release ')[-1].split(',')[0].strip().replace('.', '')
CUDA_TAG = f'cu{cuda_str}'
print(f'Torch: {TORCH_VERSION}, CUDA: {CUDA_TAG}')

# PyG sparse extensions (likely pre-installed; no-op if already present)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html

!pip install -q torch-geometric
!pip install -q networkx pandas rdkit pyemd

In [ ]:
# Cell 5: Smoke-test imports
import torch_geometric
import networkx, pandas, numpy
print('PyTorch:   ', torch.__version__)
print('PyG:       ', torch_geometric.__version__)
print('NetworkX:  ', networkx.__version__)
print('pandas:    ', pandas.__version__)
print('numpy:     ', numpy.__version__)
print('All imports OK')

In [ ]:
# Cell 6: Verify trained GNN checkpoint exists
import os
ckpt = 'param/gnns/mutag_gcn.pt'
assert os.path.exists(ckpt), f'Missing checkpoint: {ckpt} — was it included in the zip?'
print('Checkpoint found:', ckpt)

In [ ]:
# Cell 7: Sanity-check training (1 epoch)
# --verbose 1: run evaluation and save best_model.pth every epoch (needed when --epoch 1)
# mutag has tiny graphs (N~17) — no memory flags needed, bsz=32 is safe
!python main.py --dataset mutag --epoch 1 --verbose 1

In [ ]:
# Cell 8: Full training runs — memory flags depend on dataset graph size
#
# Small graphs (mutag N~17, ba3 N~25): no memory flags needed, bsz=32 is safe and fast
#   !python main.py --dataset mutag
#   !python main.py --dataset ba3
#
# Medium graphs (Tree_Cycle, BA_shapes, Tree_Grids N~30-40): bsz=16 as precaution
#   !python main.py --dataset Tree_Cycle --train_batchsize 16
#   !python main.py --dataset BA_shapes  --train_batchsize 16
#
# Large graphs (NCI1 max N~500, bbbp): full memory flags required
#   !python main.py --dataset NCI1 --gradient_checkpointing --use_amp --train_batchsize 8
#   !python main.py --dataset bbbp --gradient_checkpointing --use_amp --train_batchsize 8

!python main.py --dataset mutag